In [1]:
import cv2, mediapipe as mp
print(cv2.__version__)
print(mp.__version__)

4.12.0
0.10.14


In [6]:
import cv2
import numpy as np
import mediapipe as mp


def main():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ 웹캠을 열 수 없습니다.")
        return

    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    # 왼/오 볼 윤곽을 이루는 랜드마크 인덱스 (예시)
    left_cheek_region = [234, 93, 132, 58, 172, 136, 150, 176, 148, 152]
    right_cheek_region = [454, 323, 361, 288, 397, 366, 383, 300, 293, 334]

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("❌ 프레임을 읽을 수 없습니다.")
                break

            h, w, _ = frame.shape
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_image)

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # 1) 왼쪽 볼 polygon 만들기
                left_poly = []
                for idx in left_cheek_region:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    left_poly.append((u, v))
                    cv2.circle(frame, (u, v), 3, (0, 0, 255), -1)  # 윤곽 점 (빨강)

                left_poly_np = np.array(left_poly, np.int32)

                # 2) 왼쪽 볼 polygon 내부에 grid 샘플 찍기
                # 해상도/간격에 따라 density 조절 (예: 10픽셀 단위)
                sample_points = []
                step = 10  # 숫자 줄이면 더 촘촘해짐

                if len(left_poly) >= 3:
                    # bounding box에서만 돌면서 polygon 내부 검사
                    x_min = min(p[0] for p in left_poly)
                    x_max = max(p[0] for p in left_poly)
                    y_min = min(p[1] for p in left_poly)
                    y_max = max(p[1] for p in left_poly)

                    for u in range(x_min, x_max, step):
                        for v in range(y_min, y_max, step):
                            # polygon 내부인지 확인 (거리 모드는 필요 없어서 False)
                            dist = cv2.pointPolygonTest(left_poly_np, (u, v), False)
                            if dist >= 0:  # 내부 or 경계
                                sample_points.append((u, v))
                                # 시각화용 (연두색 작은 점)
                                cv2.circle(frame, (u, v), 2, (0, 255, 0), -1)

                # 오른쪽 볼도 동일하게 할 수 있음 (right_cheek_region 사용해서)

            cv2.imshow("Cheek Dense Sampling Demo", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        face_mesh.close()
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
